# Graph + vLLM: GREP Ψ-injection through the multi-modal channel

Runs **our** graph-PE injection scheme (`GraphAugmentedLLM`, `prism/models/gnn_llm.py`)
inside a vLLM 0.26 engine, on this Mac's torch **CPU** backend. Structure follows the
GEGR "vLLM + Graphs" tutorial, but that guide targets vLLM 0.11 and *their* PE math —
every deviation is flagged in the **Deviations** section at the bottom.

Pipeline: toy scene graph → R-PEARL Ψ (GREP chain: RMSNorm→tanh-gate) → per-prompt
tensor `[seq_len, hidden]` → vLLM multi-modal input → custom `GraphQwen2ForCausalLM`
adds `W·Ψ` post-RoPE to q/k/v in every layer.

Verified at the end: Ψ=0 is a **bitwise no-op** vs a graph-free request, and the
injection provably reaches attention (instrumented counters + amplified-Ψ probe).


In [1]:
import os

os.environ["VLLM_PLUGINS"] = ""                     # torch CPU backend, not MLX/Metal
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # in-process engine: runtime model registration

import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import Data
from transformers import AutoTokenizer, BatchFeature

from prism.models.gnn_llm import build_injection_map, node_token_variants
from prism.models.r_pearl import RandomGNNPositionalEncodings

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

/Users/jporras/.venv-vllm-metal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## A. Toy scene graph, prompt, injection map

Node-mention spans come from the repo's own `node_token_variants` +
`build_injection_map` (longest-match-first, both space/no-space tokenizations).
`scope_start=0` because this toy prompt has no ICL graphs to exclude.


In [2]:
NODE_NAMES = ["kitchen", "hallway", "garage", "bedroom", "office"]
EDGES = [(0, 1), (1, 2), (1, 3), (3, 4)]  # kitchen-hallway-garage, hallway-bedroom-office

edge_index = torch.tensor(EDGES + [(b, a) for a, b in EDGES]).T
# x is a placeholder: the random-probe path only reads its shape for num_nodes.
graph = Data(x=torch.zeros(len(NODE_NAMES), 1), edge_index=edge_index,
             num_nodes=len(NODE_NAMES))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
user_msg = (
    "Scene graph nodes: kitchen, hallway, garage, bedroom, office. "
    "You are in the kitchen. Give the shortest route to the office."
)
prompt_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": user_msg}], add_generation_prompt=True, tokenize=False
)
prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
seq_len = len(prompt_ids)

node_token_seqs = node_token_variants(NODE_NAMES, tokenizer)
injection_map = build_injection_map(prompt_ids, node_token_seqs, scope_start=0)
print(f"seq_len={seq_len}  injection_map spans per node:",
      {NODE_NAMES[n]: v for n, v in sorted(injection_map.items())})
assert set(injection_map) == set(range(len(NODE_NAMES))), "every node must be mentioned"

seq_len=57  injection_map spans per node: {'kitchen': [(28, 29), (42, 43)], 'hallway': [(30, 31)], 'garage': [(32, 33)], 'bedroom': [(34, 35)], 'office': [(36, 37), (50, 51)]}


## B. Ψ — GREP's chain, not the guide's

Mirrors `GraphAugmentedLLM.build_pe_signal`: `pe_proj(pe_model(g))` → **RMSNorm in
fp32** (weight pre-filled with the LLM's mean token-embedding RMS) → `tanh(pe_gain)`
gate → place rows at mention spans. The guide's `F.normalize(pe) * target_norm *
pe_gain` is *their* scheme; using it here would not match our training math.
The PE stack is untrained (random init, `fixed_seed_mode` for reproducibility) —
this notebook demos the *transport*, not quality. Trained weights would load into
these same modules (`pe_model.*`, `pe_proj.*`, `pe_norm.*`, `pe_gain`).

Only the token-embedding matrix is read from the checkpoint (for the RMS scale) —
the driver never loads the full LLM.


In [3]:
from huggingface_hub import snapshot_download
from safetensors import safe_open

ckpt_dir = snapshot_download(MODEL_ID, allow_patterns=["*.safetensors", "config.json"])
with safe_open(os.path.join(ckpt_dir, "model.safetensors"), framework="pt") as f:
    embed_weight = f.get_tensor("model.embed_tokens.weight").float()
HIDDEN = embed_weight.shape[1]

torch.manual_seed(0)
D_MODEL = 32
pe_model = RandomGNNPositionalEncodings(
    pe_hidden_channels=64, pe_num_layers=3, d_model=D_MODEL,
    num_samples=30, fixed_seed_mode=True,
)
pe_proj = nn.Linear(D_MODEL, HIDDEN)
pe_gain = nn.Parameter(torch.tensor(1.0))
# RMSNorm initialized to the LLM's mean token-embedding RMS (GraphAugmentedLLM.__init__)
pe_norm = nn.RMSNorm(HIDDEN)
with torch.no_grad():
    r_text = (embed_weight.norm(dim=-1).mean() / (HIDDEN ** 0.5)).item()
    pe_norm.weight.fill_(r_text)


def build_psi() -> torch.Tensor:
    """Mirror of GraphAugmentedLLM.build_pe_signal for one prompt."""
    with torch.no_grad():
        pe = pe_proj(pe_model(graph))            # [N, hidden]
        pe = pe_norm(pe.float()).to(pe.dtype)    # fp32 norm, text-scale
        pe = pe * torch.tanh(pe_gain)            # gate
        psi = torch.zeros(seq_len, HIDDEN)
        for node_idx, spans in injection_map.items():
            for start, end in spans:
                psi[start:end] += pe[node_idx]
    return psi


psi_real = build_psi()
print(f"psi: shape={tuple(psi_real.shape)}  nonzero rows={int((psi_real.abs().sum(-1) > 0).sum())}"
      f"  |psi| at nodes={psi_real.norm(dim=-1).max():.3f}  text RMS ref={r_text * HIDDEN**0.5:.3f}")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 740.00it/s]

psi: shape=(57, 896)  nonzero rows=7  |psi| at nodes=0.345  text RMS ref=0.453


## C. The vLLM 0.26 plug-in

Three processor classes + the wrapper model. Key 0.26 facts (vs the guide's 0.11):

- The data parser only speaks image/video/audio, so the graph tensor rides under the
  `"image"` modality label as `{"image": {"graph_embeds": tensor}}` (same trick as
  vLLM's in-tree `terratorch` model). Cosmetic only.
- We submit **real token ids** — no all-placeholder prompt, no carrying X. The mm
  tensor is Ψ alone (`[seq_len, hidden]`, half the guide's payload); the engine
  computes token embeddings itself, exactly like training-time `_augment_embeddings`.
  `PlaceholderRange(offset=0, length=seq_len)` still declares full-prompt coverage.
- Profiling still must run at **full seq_len** (the guide's silent-hang bug class,
  vllm#26223) — see `GraphDummyInputsBuilder`.


In [4]:
from vllm import LLM, ModelRegistry, SamplingParams
from vllm.config import VllmConfig
from vllm.inputs import MultiModalInput, mm_input
from vllm.model_executor.models.interfaces import SupportsMultiModal
from vllm.model_executor.models.utils import (
    AutoWeightsLoader, WeightsMapper, init_vllm_registered_model, maybe_prefix,
)
from vllm.multimodal import MULTIMODAL_REGISTRY
from vllm.multimodal.inputs import (
    MultiModalFieldConfig, MultiModalKwargsItems, PlaceholderRange,
)
from vllm.multimodal.parse import DictEmbeddingItems, MultiModalDataParser
from vllm.multimodal.processing import (
    BaseDummyInputsBuilder, BaseMultiModalProcessor, BaseProcessingInfo,
)

# vLLM's data parser only speaks image/video/audio, so the graph tensor rides
# under the "image" modality label (terratorch does the same). Cosmetic only.
_FIELDS = {"graph_embeds": "image"}


def _graph_fields_config(hf_inputs=None):
    return {name: MultiModalFieldConfig.batched(mod) for name, mod in _FIELDS.items()}


class GraphDataParser(MultiModalDataParser):
    def _parse_image_data(self, data):
        if isinstance(data, dict):
            return DictEmbeddingItems(
                data, modality="image",
                required_fields=set(_FIELDS),
                fields_factory=_graph_fields_config,
            )
        return super()._parse_image_data(data)


class GraphProcessingInfo(BaseProcessingInfo):
    def get_data_parser(self):
        return GraphDataParser()

    def get_supported_mm_limits(self):
        return {"image": 1}


class GraphDummyInputsBuilder(BaseDummyInputsBuilder):
    def get_dummy_text(self, mm_counts):
        return ""

    def get_dummy_mm_data(self, seq_len, mm_counts, mm_options=None):
        # Profile at FULL seq_len: a runtime item larger than the profiled one
        # makes the v1 scheduler retry forever (vllm#26223 family).
        hidden = self.info.get_hf_config().hidden_size
        return {"image": {"graph_embeds": torch.zeros(1, seq_len, hidden)}}


class GraphMultiModalProcessor(BaseMultiModalProcessor):
    def _get_mm_fields_config(self, hf_inputs, hf_processor_mm_kwargs):
        return _graph_fields_config(hf_inputs)

    def _get_prompt_updates(self, mm_items, hf_processor_mm_kwargs, out_mm_kwargs):
        return []

    def apply(self, inputs, timing_ctx) -> MultiModalInput:
        _, passthrough = self._get_hf_mm_data(inputs.mm_data_items)
        g = torch.as_tensor(passthrough["graph_embeds"])
        if g.ndim == 2:
            g = g.unsqueeze(0)
        rows = g.shape[1]

        mm_kwargs = MultiModalKwargsItems.from_hf_inputs(
            BatchFeature({"graph_embeds": g}, tensor_type="pt"),
            self._get_mm_fields_config(None, {}),
        )
        mm_hashes = inputs.get_mm_hashes(self.info.model_id)

        prompt = inputs.prompt
        if isinstance(prompt, str):
            # Profiling path (dummy text is ""): synthesize a full-length prompt.
            prompt_ids = [self.info.get_tokenizer().eos_token_id] * rows
        else:
            prompt_ids = list(prompt)
            if len(prompt_ids) != rows:
                raise ValueError(
                    f"graph mm rows ({rows}) != prompt length ({len(prompt_ids)}); "
                    "psi and prompt_token_ids must come from the same tokenization"
                )

        return mm_input(
            prompt_token_ids=prompt_ids,
            mm_kwargs=mm_kwargs,
            mm_hashes=mm_hashes,
            mm_placeholders={"image": [PlaceholderRange(offset=0, length=rows)]},
        )

INFO 08-07 18:06:13 [__init__.py:52] Available plugins for group vllm.platform_plugins:


INFO 08-07 18:06:13 [__init__.py:54] - metal -> vllm_metal:register


INFO 08-07 18:06:13 [importing.py:88] Triton not installed or not compatible; certain GPU-related functions will not be available.


### The wrapper model

- `embed_input_ids(input_ids, multimodal_embeddings, is_multimodal)` replaces the
  guide's `get_input_embeddings` hook: the v1 runner calls it **every scheduling
  step with the exact packed token batch** and an alignment mask, so the
  "stash Ψ on the instance" trick stays batch-aligned — including chunked prefill
  and mixed continuous batches. Decode steps arrive with no mm rows → stash stays
  `None` → stock attention.
- Qwen2.5 has q/k/v **biases**. Our `GraphAugmentedLLM` refuses such models
  (Gemma is bias-free); here we instead compute `F.linear(Ψ, W)` weight-only, so
  Ψ=0 rows contribute *exactly* zero — no need for the guide's `f(Ψ) − f(0)`
  double projection.
- The `dbg` counters are cheap and stay in: they are what caught the difference
  between "output unchanged because Ψ is weak" and "output unchanged because the
  channel is dead".


In [5]:
@MULTIMODAL_REGISTRY.register_processor(
    GraphMultiModalProcessor,
    info=GraphProcessingInfo,
    dummy_inputs=GraphDummyInputsBuilder,
)
class GraphQwen2ForCausalLM(nn.Module, SupportsMultiModal):
    hf_to_vllm_mapper = WeightsMapper(orig_to_new_prefix={"": "language_model."})

    @classmethod
    def get_placeholder_str(cls, modality, i):
        return None

    def __init__(self, *, vllm_config: VllmConfig, prefix: str = ""):
        super().__init__()
        self.config = vllm_config.model_config.hf_config
        self.language_model = init_vllm_registered_model(
            vllm_config=vllm_config,
            prefix=maybe_prefix(prefix, "language_model"),
            architectures=["Qwen2ForCausalLM"],
        )
        # psi for the CURRENT packed token batch; armed in embed_input_ids,
        # read by the patched attention forwards, cleared after forward.
        self._psi_packed: torch.Tensor | None = None
        self.dbg = {"embed_calls": 0, "psi_armed": 0, "attn_hit": 0, "attn_skip_shape": 0}
        self._install_psi_injection()

    def _install_psi_injection(self):
        wrapper = self
        for layer in self.language_model.model.layers:
            attn = layer.self_attn
            if getattr(attn, "qk_norm", False):
                raise ValueError("qk-normed attention is not handled by this demo patch")

            def make_forward(attn):
                def forward(positions, hidden_states):
                    qkv, _ = attn.qkv_proj(hidden_states)
                    q, k, v = qkv.split([attn.q_size, attn.kv_size, attn.kv_size], dim=-1)
                    q, k = attn.rotary_emb(positions, q, k)
                    psi = wrapper._psi_packed
                    if psi is not None and psi.shape[0] != hidden_states.shape[0]:
                        wrapper.dbg["attn_skip_shape"] += 1
                    if psi is not None and psi.shape[0] == hidden_states.shape[0]:
                        wrapper.dbg["attn_hit"] += 1
                        # Weight-only linear: Qwen2's qkv_proj has a bias, and psi
                        # must contribute exactly 0 at psi=0 rows (GREP invariant;
                        # GraphAugmentedLLM enforces bias-free projections instead).
                        w = attn.qkv_proj.weight
                        qkv_psi = F.linear(psi.to(dtype=w.dtype, device=w.device), w)
                        q_psi, k_psi, v_psi = qkv_psi.split(
                            [attn.q_size, attn.kv_size, attn.kv_size], dim=-1)
                        # Post-RoPE, mirroring _prism_pe_attention_forward:
                        # q = RoPE(W_q h) + W_q psi ; k likewise ; v = W_v h + W_v psi
                        q = q + q_psi
                        k = k + k_psi
                        v = v + v_psi
                    attn_output = attn.attn(q, k, v)
                    output, _ = attn.o_proj(attn_output)
                    return output

                return forward

            attn.forward = make_forward(attn)

    def embed_multimodal(self, **kwargs):
        g = kwargs.get("graph_embeds")
        if g is None:
            return []
        if isinstance(g, torch.Tensor):
            return tuple(g[i] for i in range(g.shape[0]))
        return tuple(torch.as_tensor(t) for t in g)

    def embed_input_ids(self, input_ids, multimodal_embeddings=None, *, is_multimodal=None):
        embeds = self.language_model.embed_input_ids(input_ids)
        self._psi_packed = None
        self.dbg["embed_calls"] += 1
        if multimodal_embeddings is not None and len(multimodal_embeddings) > 0:
            self.dbg["psi_armed"] += 1
            rows = torch.cat(list(multimodal_embeddings), dim=0)
            n_flagged = int(is_multimodal.sum().item())
            if n_flagged != rows.shape[0]:
                raise ValueError(
                    f"psi rows ({rows.shape[0]}) != flagged positions ({n_flagged})")
            psi = torch.zeros_like(embeds)
            psi[is_multimodal] = rows.to(dtype=embeds.dtype, device=embeds.device)
            self._psi_packed = psi
        return embeds

    def forward(self, input_ids, positions, intermediate_tensors=None,
                inputs_embeds=None, **kwargs):
        try:
            return self.language_model(
                input_ids, positions, intermediate_tensors, inputs_embeds)
        finally:
            self._psi_packed = None

    def compute_logits(self, *args, **kwargs):
        return self.language_model.compute_logits(*args, **kwargs)

    def load_weights(self, weights):
        loader = AutoWeightsLoader(self)
        return loader.load_weights(weights, mapper=self.hf_to_vllm_mapper)

    def get_language_model(self):
        return self.language_model


ModelRegistry.register_model("GraphQwen2ForCausalLM", GraphQwen2ForCausalLM)

## D. Engine + verification

CPU-backend notes: `VLLM_PLUGINS=""` keeps the MLX/Metal plugin out (its model
forward is MLX — torch-level patches never execute there); `gpu_memory_utilization`
is a **RAM** reservation on this backend; `enable_mm_embeds=True` is required to
submit precomputed embeddings; `enforce_eager=True` keeps the Python-state patch
live (same reason as the guide). `VLLM_ENABLE_V1_MULTIPROCESSING=0` (set in the
first cell) keeps the engine in-process so the runtime model registration is seen.


In [6]:
llm = LLM(
    model=MODEL_ID,
    hf_overrides={"architectures": ["GraphQwen2ForCausalLM"]},
    dtype="float32",
    enforce_eager=True,
    max_model_len=1024,
    gpu_memory_utilization=0.25,  # CPU backend: fraction of RAM to reserve
    enable_mm_embeds=True,        # we pass precomputed embeddings, not raw media
    enable_prefix_caching=False,
    disable_log_stats=True,
)

INFO 08-07 18:06:14 [api_utils.py:273] non-default args: {'dtype': 'float32', 'max_model_len': 1024, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.25, 'disable_log_stats': True, 'hf_overrides': {'architectures': ['GraphQwen2ForCausalLM']}, 'enforce_eager': True, 'enable_mm_embeds': True, 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 08-07 18:06:15 [arg_utils.py:1631] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 08-07 18:06:16 [model.py:623] Resolved architecture: GraphQwen2ForCausalLM


INFO 08-07 18:06:16 [model.py:2117] Upcasting torch.bfloat16 to torch.float32.


INFO 08-07 18:06:16 [model.py:1788] Using max model len 1024


...

INFO 08-07 18:06:26 [kv_cache_utils.py:2177] GPU KV cache size: 77,696 tokens


INFO 08-07 18:06:26 [kv_cache_utils.py:2178] Maximum concurrency for 1,024 tokens per request: 75.88x


INFO 08-07 18:06:26 [core.py:347] init engine (profile, create kv cache, warmup model) took 5.16 s


WARNING 08-07 18:06:26 [cpu_worker.py:153] CPU backend doesn't allow to use `torch.set_num_threads` after the thread binding, skip it.


WARNING 08-07 18:06:26 [cpu_worker.py:153] CPU backend doesn't allow to use `torch.set_num_threads` after the thread binding, skip it.


In [7]:
sp = SamplingParams(temperature=0, max_tokens=48)


def gen(psi=None):
    req = {"prompt_token_ids": prompt_ids}
    if psi is not None:
        req["multi_modal_data"] = {"image": {"graph_embeds": psi.unsqueeze(0)}}
    return llm.generate([req], sp)[0].outputs[0]


model = llm.llm_engine.model_executor.driver_worker.worker.model_runner.model

out_base = gen()
dbg_base = dict(model.dbg)
out_zero = gen(torch.zeros_like(psi_real))
out_real = gen(psi_real)
out_amp = gen(psi_real * 20)
print("\ndbg after baseline:", dbg_base, " final:", model.dbg)

print("\nbaseline :", repr(out_base.text))
print("psi zero :", repr(out_zero.text))
print("psi real :", repr(out_real.text))
print("psi x20  :", repr(out_amp.text))

assert out_zero.token_ids == out_base.token_ids, "psi=0 must be a bitwise no-op!"
print("\nPASS: psi=0 request identical to graph-free baseline")
assert model.dbg["psi_armed"] > 0, "psi never armed — mm data not reaching the model!"
assert model.dbg["attn_hit"] > 0, "psi armed but attention patch never consumed it!"
print(f"psi real differs from baseline: {out_real.token_ids != out_base.token_ids}")
print(f"psi x20  differs from baseline: {out_amp.token_ids != out_base.token_ids}")
print("SMOKE TEST COMPLETE")

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00, 178.70it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.01it/s, est. speed input: 57.37 toks/s, output: 48.31 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.01it/s, est. speed input: 57.37 toks/s, output: 48.31 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s, est. speed input: 57.94 toks/s, output: 48.79 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s, est. speed input: 57.94 toks/s, output: 48.79 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s, est. speed input: 57.94 toks/s, output: 48.79 toks/s]


dbg after baseline: {'embed_calls': 48, 'psi_armed': 0, 'attn_hit': 0, 'attn_skip_shape': 0}  final: {'embed_calls': 192, 'psi_armed': 3, 'attn_hit': 72, 'attn_skip_shape': 0}

baseline : "To get to the office from the kitchen, the shortest route would be through the hallway. Here's the step-by-step reasoning:\n\n1. **Kitchen**: The kitchen is the immediate starting point.\n2. **Hallway**: The hallway is"
psi zero : "To get to the office from the kitchen, the shortest route would be through the hallway. Here's the step-by-step reasoning:\n\n1. **Kitchen**: The kitchen is the immediate starting point.\n2. **Hallway**: The hallway is"
psi real : "To get to the office from the kitchen, the shortest route would be through the hallway. Here's the step-by-step reasoning:\n\n1. **Kitchen**: The kitchen is the immediate starting point.\n2. **Hallway**: The hallway is"
psi x20  : 'To get to the office from the kitchen, you can follow these steps:\n\n1. **Enter the kitchen**: Start by entering 

## Deviations from the GEGR guide, and what transfers

| # | Guide (vLLM 0.11, GEGR) | Here (vLLM 0.26, GREP) |
|---|---|---|
| 1 | `get_input_embeddings` hook | `embed_input_ids(..., multimodal_embeddings, is_multimodal)` — runner supplies the batch-alignment mask |
| 2 | Whole prompt = placeholder tokens; tensor carries `[X ‖ Ψ]` (`2·hidden`) | Real token ids; tensor is Ψ only (`hidden`) — engine computes X |
| 3 | Custom `"graph"` modality | `"image"` label, dict payload (`terratorch` pattern) + `enable_mm_embeds=True` |
| 4 | Ψ scale: `normalize · target_norm · gain` | GREP chain: fp32 RMSNorm (text-RMS init) · `tanh(pe_gain)` |
| 5 | Bias handled by `f(Ψ) − f(0)` | Weight-only `F.linear(Ψ, W)`; note our HF-side `GraphAugmentedLLM` instead *refuses* biased projections (Gemma is bias-free) |
| 6 | CUDA GPU | torch CPU backend (`VLLM_PLUGINS=""`); the Metal/MLX fast path cannot take torch patches at all |

**Still true from the guide** (verified or by construction): profile dummy inputs at
full seq_len; `enforce_eager` required; Ψ is prompt-side only — tokens generated at
decode time get Ψ=0, unlike the HF decode path's `NodeMentionMatcher`; precompute
outside the engine, LLM inside.

**Toward betty/CUDA:** the plug-in classes transfer as-is (CUDA just makes them
fast); Gemma needs the same patch written against vLLM's Gemma attention (separate
q/k/v projections, KV-shared layers — mirror `_prism_pe_attention_forward`'s
`is_kv_shared_layer` skip); load trained PE weights into the section-B modules; for
multiprocess engines register the model as a package plugin entry point instead of
setting `VLLM_ENABLE_V1_MULTIPROCESSING=0`.
